<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import numpy as np
import pandas as pd

# Load the public anonymized starter data from this repository.
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "sunainakhatwani12/flyrank-ml-internship-sunaina/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Raw data shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

# Safe numeric feature candidates.
numeric_candidates = [
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

# Safe categorical feature candidates.
categorical_candidates = [
    "content_type"
]

numeric_features = [
    column for column in numeric_candidates
    if column in df.columns
]

categorical_features = [
    column for column in categorical_candidates
    if column in df.columns
]

if not numeric_features:
    raise ValueError(
        "None of the expected numeric features were found. "
        "Check the printed column names above."
    )

# Start with safe, pre-decision features only.
X = df[numeric_features + categorical_features].copy()

# Convert numeric fields and fill missing values with the median.
for column in numeric_features:
    X[column] = pd.to_numeric(X[column], errors="coerce")
    X[column] = X[column].fillna(X[column].median())

# Add log versions of skewed count features when available.
if "impressions_90d" in X.columns:
    X["log_impressions_90d"] = np.log1p(X["impressions_90d"])

if "clicks_90d" in X.columns:
    X["log_clicks_90d"] = np.log1p(X["clicks_90d"])

# Fill and encode categorical fields.
for column in categorical_features:
    X[column] = X[column].fillna("Unknown").astype(str)

if categorical_features:
    X = pd.get_dummies(
        X,
        columns=categorical_features,
        prefix=categorical_features,
        dtype=int
    )

feature_vector = X

print("\nFeature-vector shape:", feature_vector.shape)
print("\nFinal feature columns:")
print(feature_vector.columns.tolist())

print("\nMissing values remaining:")
print(feature_vector.isna().sum().sort_values(ascending=False).head(10))

feature_vector.head()

Raw data shape: (30000, 44)
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Feature-vector shape: (30000, 11)

Final feature columns:
['content_age_days', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'log_impressions_90d', 'log

,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,log_impressions_90d,log_clicks_90d,content_type_comparison article,content_type_feedly article,content_type_keyword article
0,187,3803,29,0.76,10.6,5.88,8.243808,3.401197,0,0,1
1,445,15320,7,0.05,20.3,0.00,9.636980,2.079442,0,0,1
2,141,12581,11,0.09,36.5,0.00,9.440023,2.484907,0,0,1
3,463,11751,58,0.49,6.2,1.28,9.371779,4.077537,0,0,1
4,263,19140,24,0.13,44.0,0.00,9.859588,3.218876,0,0,1


### Feature vector design

The unit of analysis is one content page.

I built the feature vector using historical content and search-performance signals that would be available before making a refresh-priority decision.

The main feature groups are:

- Content age
- Search visibility
- Click performance
- Average search position
- Engagement
- Content type, when available

Numeric missing values are filled using the median. Categorical missing values are filled with `Unknown` and converted into one-hot encoded columns.

Label-derived fields, client identifiers, and future information are not included in the feature vector.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
print("Feature Summary\n")

for column in feature_vector.columns:
    print(f"{column:30} Missing values: {feature_vector[column].isna().sum()}")

Feature Summary

content_age_days               Missing values: 0
impressions_90d                Missing values: 0
clicks_90d                     Missing values: 0
ctr                            Missing values: 0
avg_position                   Missing values: 0
engagement_rate                Missing values: 0
log_impressions_90d            Missing values: 0
log_clicks_90d                 Missing values: 0
content_type_comparison article Missing values: 0
content_type_feedly article    Missing values: 0
content_type_keyword article   Missing values: 0


### Feature notes

| Feature | Meaning | Missing value handling | Available before prediction? |
|---------|---------|------------------------|------------------------------|
| content_age_days | Age of the content page | Filled with median | Yes |
| impressions_90d | Search impressions during the previous 90 days | Filled with median | Yes |
| clicks_90d | Search clicks during the previous 90 days | Filled with median | Yes |
| ctr | Click-through rate | Filled with median | Yes |
| avg_position | Average search ranking position | Filled with median | Yes |
| engagement_rate | User engagement signal | Filled with median | Yes |
| content_type | Type of content | Missing values replaced with "Unknown" and one-hot encoded | Yes |

All selected features exist before the content refresh decision is made, making them appropriate for prediction.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Section 3 — Leakage hunt

known_leakage_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "target",
    "label"
}

identifier_columns = {
    "client_id",
    "content_id"
}

future_or_post_action_columns = {
    "refresh_result",
    "post_refresh_clicks",
    "post_refresh_impressions",
    "future_ctr",
    "future_position"
}

selected_columns = set(feature_vector.columns)

found_label_leakage = sorted(selected_columns & known_leakage_columns)
found_identifiers = sorted(selected_columns & identifier_columns)
found_future_fields = sorted(selected_columns & future_or_post_action_columns)

print("LEAKAGE AUDIT")
print("-" * 60)

print("Label-derived fields found:", found_label_leakage)
print("Identifiers found:", found_identifiers)
print("Future/post-action fields found:", found_future_fields)

all_risky_found = (
    found_label_leakage
    + found_identifiers
    + found_future_fields
)

if all_risky_found:
    raise ValueError(
        f"Leakage-risk columns found in feature vector: {all_risky_found}"
    )

print("\nPASS: No known leakage-risk columns are present.")
print("Client and content IDs remain available only outside the feature vector.")

LEAKAGE AUDIT
------------------------------------------------------------
Label-derived fields found: []
Identifiers found: []
Future/post-action fields found: []

PASS: No known leakage-risk columns are present.
Client and content IDs remain available only outside the feature vector.


### Leakage hunt

I checked the feature vector for three major leakage risks:

1. **Label-derived fields**  
   Columns such as `trend_direction`, `trend_pct`, and the target label directly describe the outcome, so they must not be used as model features.

2. **Identifiers**  
   `client_id` and `content_id` identify groups or rows. They may be useful for grouped splitting or record tracking, but they should not be predictive features.

3. **Future or post-decision information**  
   Any field created after the prediction moment, including future outcomes or action results, would leak information that would not be available when the recommendation is generated.

The test below checks whether any known risky columns accidentally appear in the final feature vector.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
excluded_fields = {
    "trend_direction": "Label-derived field",
    "trend_pct": "Label-derived field",
    "client_id": "Grouping only",
    "content_id": "Identifier only",
    "future metrics": "Not available before prediction"
}

print("Excluded fields\n")

for field, reason in excluded_fields.items():
    print(f"{field:20} -> {reason}")

Excluded fields

trend_direction      -> Label-derived field
trend_pct            -> Label-derived field
client_id            -> Grouping only
content_id           -> Identifier only
future metrics       -> Not available before prediction


### Fields excluded from the model

The following fields were intentionally excluded from the feature vector:

| Field | Reason |
|-------|--------|
| trend_direction | Label-derived field that directly describes the outcome. |
| trend_pct | Label-derived field that leaks target information. |
| client_id | Used only for grouped train/test splitting, never as a feature. |
| content_id | Identifier only; provides no meaningful predictive signal. |
| Any future or post-refresh metrics | Not available at prediction time and would introduce information leakage. |

These exclusions help ensure that the model learns only from historical information that would realistically be available when recommending pages for review.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.